In [2]:
from openai import OpenAI
import requests
import json

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

modeles_installes = [m["name"] for m in requests.get("http://localhost:11434/api/tags").json()["models"]]
print(modeles_installes)

MODEL = "qwen3-vl:8b-instruct"

['qwen2.5vl:7b', 'qwen3-vl:8b-instruct', 'qwen3:4b', 'qwen3:8b']


In [3]:
import base64
import time
from pathlib import Path

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 1 et 2 : le fichier devient du texte base64
photo = Path("data/photos/10_3608580065340_etiquette.jpg")
image_b64 = base64.b64encode(photo.read_bytes()).decode("ascii")

# 3 : l'adresse data: (le type doit correspondre au format du fichier)
url_image = f"data:image/jpeg;base64,{image_b64}"

# 4 : un message en deux morceaux, texte puis image
messages = [{"role": "user", "content": [
    {"type": "text", "text": "Recopie le tableau nutritionnel de cette photo."},
    {"type": "image_url", "image_url": {"url": url_image}},
]}]

debut = time.perf_counter()
reponse = client.chat.completions.create(
    model="qwen3-vl:8b-instruct", messages=messages, temperature=0
)
print(reponse.choices[0].message.content)
print(f"{time.perf_counter() - debut:.1f} s,",
      reponse.usage.prompt_tokens, "tokens en entrée,",
      reponse.usage.completion_tokens, "en sortie")

Voici le tableau nutritionnel recopié à partir de l'image :

**Valeurs nutritionnelles moyennes pour 100 g :**

- **Energie** : 2299 kJ / 551 kcal
- **Matières grasses** : 34 g  
  *dont acides gras saturés* : 5,2 g
- **Glucides** : 53 g  
  *dont sucres* : 51 g
- **Fibres alimentaires** : 3,3 g
- **Protéines** : 6,7 g
- **Sel** : 0,10 g

*(Note : Le nom du produit n’est pas visible sur l’image, mais le logo "BONNE MAMAN" apparaît en bas.)*
16.3 s, 1058 tokens en entrée, 181 en sortie
